In [240]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [241]:
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import matplotlib
import seaborn as sns
import copy
import pickle
import warnings
warnings.filterwarnings('ignore')
import shap
from models import *
from utilities import *

In [242]:
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42 
matplotlib.rc('font', family='sans-serif') 
matplotlib.rc('font', serif='Helvetica Neue') 

In [243]:
##########################################################################################################

In [244]:
import tensorflow as tf
tf.compat.v1.disable_v2_behavior()

In [245]:
shap.explainers._deep.deep_tf.op_handlers["AddV2"] = shap.explainers._deep.deep_tf.passthrough 
shap.explainers._deep.deep_tf.op_handlers["FusedBatchNormV3"] = shap.explainers._deep.deep_tf.passthrough

In [246]:
################### data import ###############

In [247]:
data_dir = '../results/data/'
data_copy = pd.read_pickle(data_dir + 'LSTM_processed_data_dy_filter.pkl')
ba_copy = pd.read_pickle(data_dir + 'LSTM_processed_data_ba_filter.pkl')
data_copy['Id_encoded'], uniques = pd.factorize(data_copy['SCCRIP_ID'])
mapping_dict = {value: index for index, value in enumerate(uniques)}

In [248]:
ba_copy['Id_encoded'] = ba_copy['SCCRIP_ID'].map(mapping_dict)
ba_copy = ba_copy[ba_copy['Id_encoded'].notna()]
ba_copy = ba_copy.drop_duplicates(subset='SCCRIP_ID', keep='last')
ba_copy.drop('SCCRIP_ID', inplace=True, axis=1)

In [249]:
ba_copy = ba_copy.fillna(0)
data_copy = data_copy.fillna(0)

In [250]:
with open(data_dir +  'SCCRIP_ID_dict.pkl', 'wb') as f:
    pickle.dump(mapping_dict, f)

In [251]:
len(data_copy.SCCRIP_ID.unique())

1267

In [252]:
################ data process ##################

In [253]:
import pickle, gzip
from pathlib import Path

def save_y_and_params(y_dict, params, out_path, compress=True):
    """
    y_dict: e.g. {"train": y_train, "val": y_val, "test": y_test}
    params: dict of hyperparameters/config
    out_path: path to .pkl or .pkl.gz
    """
    payload = {"y": y_dict, "params": params}
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    opener = gzip.open if compress else open
    with opener(out_path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_y_and_params(path, compressed=True):
    opener = gzip.open if compressed else open
    with opener(path, "rb") as f:
        payload = pickle.load(f)
    return payload["y"], payload["params"]


In [254]:
# seq_length = 3

In [255]:
s_l = [3,4,5,6,7]

for seq_length in s_l:
    ys = {'pred':[],
         'test':[]}
    paras = {'ls':[],
             'dt':[],
             'r2':[]}
    r = 0
    while r < 100:
        print(seq_length, r)
        sequences, seq_ba, labels, date,  y_target = create_dataset(data_copy, ba_copy, seq_length, mapping_dict)

        sequences_copy, seq_ba_copy, labels_copy, date_copy, y_target_copy = shuffle_dataset(sequences, seq_ba, labels, date, y_target)

        X, y, X_bg, dt, ls, seq_map = create_sequences_with_background(sequences_copy, labels_copy, seq_ba_copy, date_copy,y_target_copy, seq_length)
        X_normalized, X_bg_normalized = normalize_X(X, X_bg)

        (X_train,X_val,X_test,
                X_bg_train,X_bg_val,X_bg_test,
                dt_train, dt_val, dt_test, 
                y_train,y_val,y_test,
                ls_train, ls_val, ls_test,
                seq_map_train, seq_map_val, seq_map_test) = split_dataset(X_normalized, X_bg_normalized,dt, y, ls, seq_map)

        combined_model = CombinedLSTMModel(seq_length=seq_length, n_features=int(X_train.shape[2]), bg_features=int(X_bg_train.shape[2]))
        # Train the model
        history = combined_model.train(
            X_train, X_bg_train, dt_train, y_train,
            X_val,   X_bg_val,   dt_val,   y_val,
            epochs=20, batch_size=64,
            checkpoint_path="../results/models/mix_best_t.h5"
        )

        y_pred = combined_model.predict(X_test, X_bg_test, dt_test)
        r2 = r2_score(y_test.flatten(), y_pred.flatten())
        if r2 > 0:
            ys['pred'].append(copy.deepcopy(y_pred))
            ys['test'].append(copy.deepcopy(y_test))

            paras['ls'].append(copy.deepcopy(ls_test))
            paras['dt'].append(copy.deepcopy(dt_test))
            paras['r2'].append(copy.deepcopy(r2))
            
            r += 1
    fname = f"../results/data/mix/r100_y_and_params{seq_length}.pkl"
    save_y_and_params(ys, paras, fname, compress=False)

7 0
Model input shapes: (?, 7, 868) (?, 1, 44) (?, 1, 1)
Model: "CombinedLSTMModel"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_bg (InputLayer)       [(None, 1, 44)]              0         []                            
                                                                                                  
 input_dt (InputLayer)       [(None, 1, 1)]               0         []                            
                                                                                                  
 input_seq (InputLayer)      [(None, 7, 868)]             0         []                            
                                                                                                  
 bg_plus_dt (Concatenate)    (None, 1, 45)                0         ['input_bg[0][0]',            
                         

2025-09-24 11:38:57.929724: W tensorflow/c/c_api.cc:305] Operation '{name:'training_248/Adam/beta_2/Assign' id:190652 op device:{requested: '', assigned: ''} def:{{{node training_248/Adam/beta_2/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](training_248/Adam/beta_2, training_248/Adam/beta_2/Initializer/initial_value)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.


Epoch 1/20


2025-09-24 11:39:14.898958: W tensorflow/c/c_api.cc:305] Operation '{name:'loss_248/mul' id:190282 op device:{requested: '', assigned: ''} def:{{{node loss_248/mul}} = Mul[T=DT_FLOAT, _has_manual_control_dependencies=true](loss_248/mul/x, loss_248/relu_out_loss/value)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.



Epoch 1: val_loss improved from inf to 17.51604, saving model to ../results/models/mix_best_t.h5
Epoch 2/20

Epoch 2: val_loss improved from 17.51604 to 13.59876, saving model to ../results/models/mix_best_t.h5
Epoch 3/20

Epoch 3: val_loss did not improve from 13.59876
Epoch 4/20

Epoch 4: val_loss improved from 13.59876 to 12.70376, saving model to ../results/models/mix_best_t.h5
Epoch 5/20

Epoch 5: val_loss improved from 12.70376 to 12.11429, saving model to ../results/models/mix_best_t.h5
Epoch 6/20

Epoch 6: val_loss did not improve from 12.11429
Epoch 7/20

Epoch 7: val_loss did not improve from 12.11429
Epoch 8/20

Epoch 8: val_loss did not improve from 12.11429
Epoch 9/20

Epoch 9: val_loss did not improve from 12.11429
Epoch 10/20

Epoch 10: val_loss did not improve from 12.11429
Epoch 11/20

Epoch 11: val_loss did not improve from 12.11429
Epoch 12/20

Epoch 12: val_loss did not improve from 12.11429
Epoch 13/20

Epoch 13: val_loss did not improve from 12.11429
Epoch 14/20


...

2025-09-24 20:59:40.290518: W tensorflow/c/c_api.cc:305] Operation '{name:'loss_446/mul' id:341950 op device:{requested: '', assigned: ''} def:{{{node loss_446/mul}} = Mul[T=DT_FLOAT, _has_manual_control_dependencies=true](loss_446/mul/x, loss_446/relu_out_loss/value)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.



Epoch 1: val_loss improved from inf to 21.31319, saving model to ../results/models/mix_best_t.h5
Epoch 2/20

Epoch 2: val_loss improved from 21.31319 to 16.72691, saving model to ../results/models/mix_best_t.h5
Epoch 3/20

Epoch 3: val_loss improved from 16.72691 to 14.95328, saving model to ../results/models/mix_best_t.h5
Epoch 4/20

Epoch 4: val_loss did not improve from 14.95328
Epoch 5/20

Epoch 5: val_loss improved from 14.95328 to 13.52720, saving model to ../results/models/mix_best_t.h5
Epoch 6/20

Epoch 6: val_loss did not improve from 13.52720
Epoch 7/20

Epoch 7: val_loss did not improve from 13.52720
Epoch 8/20

Epoch 8: val_loss did not improve from 13.52720
Epoch 9/20

Epoch 9: val_loss improved from 13.52720 to 13.29213, saving model to ../results/models/mix_best_t.h5
Epoch 10/20

Epoch 10: val_loss did not improve from 13.29213
Epoch 11/20

Epoch 11: val_loss did not improve from 13.29213
Epoch 12/20

Epoch 12: val_loss did not improve from 13.29213
Epoch 13/20

Epoch 1

2025-09-24 21:03:04.403098: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_16_447/bias/Assign' id:342801 op device:{requested: '', assigned: ''} def:{{{node dense_16_447/bias/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](dense_16_447/bias, dense_16_447/bias/Initializer/zeros)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.
2025-09-24 21:03:37.310538: W tensorflow/c/c_api.cc:305] Operation '{name:'dense_128_447/bias/m/Assign' id:342983 op device:{requested: '', assigned: ''} def:{{{node dense_128_447/bias/m/Assign}} = AssignVariableOp[_has_manual_control_dependencies=true, dtype=DT_FLOAT, validate_shape=false](dense_128_447/bias/m, dense_128_447/bias/m/Initializer/zeros)}}' was changed by setting attribute after it was run by a session. This mutation will have 

Best model loaded from: ../results/models/mix_best_t.h5


2025-09-24 21:04:10.525179: W tensorflow/c/c_api.cc:305] Operation '{name:'relu_out_447/Relu' id:342827 op device:{requested: '', assigned: ''} def:{{{node relu_out_447/Relu}} = Relu[T=DT_FLOAT, _has_manual_control_dependencies=true](dense_out_447/BiasAdd)}}' was changed by setting attribute after it was run by a session. This mutation will have no effect, and will trigger an error in the future. Either don't modify nodes after running them or create a new session.


In [256]:
# save_y_and_params(ys, paras, "../results/data/mix/y_and_params3.pkl", compress=False)